# 45. The Terminal Digital Twin Scoping Problem
## Tier 1: The Pen & Paper Method (Mathematical Formulation)

### Goal
Formulate the Terminal Digital Twin Scoping Problem as a multi-objective integer programming model that balances coverage maximization, cost minimization, and operational value optimization.

### Key assumptions
- Each terminal component can be either included (1) or excluded (0) from the digital twin
- Sensors must be placed to provide adequate coverage for selected components
- Budget, resource, and complexity constraints limit the scope of implementation
- Components may have interdependencies that affect selection decisions

### Approach (step-by-step)
1. Define decision variables for component selection and sensor placement
2. Formulate objective function to maximize value-to-cost ratio
3. Add constraints for coverage, budget, resources, and complexity
4. Include interdependency constraints for critical component pairs
5. Solve the mixed-integer programming model using exact methods

### What to look for in the results
- Optimal component selection that maximizes operational value within constraints
- Sensor placement strategy that ensures adequate coverage
- Trade-off analysis between value, cost, and system complexity
- Sensitivity analysis showing how parameter changes affect optimal solutions

### Concrete example (from the source)
Mediterranean Container Terminal (MCT) scenario with:
- 3 potential components: Quay Crane, Yard Block, Gate System
- 4 potential sensor locations
- Budget constraint: $250,000
- Resource constraint: 80 person-hours
- Component values: [85, 65, 75]
- Component costs: [120, 80, 95]
- Complexity factors: [3, 2, 4]
- Resource requirements: [40, 25, 35]

In [1]:
# Import required packages for mathematical optimization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

In [2]:
# Define component data structure for the terminal digital twin scoping problem
class Component:
    """Represents a terminal component that can be included in the digital twin"""
    def __init__(self, id, name, value, cost, complexity, resources):
        self.id = id
        self.name = name
        self.value = value  # Operational value score
        self.cost = cost    # Implementation cost in thousands
        self.complexity = complexity  # System complexity factor
        self.resources = resources  # Resource requirements (person-hours)
    
    def __repr__(self):
        return f"Component({self.id}: {self.name}, V={self.value}, C={self.cost}, D={self.complexity}, R={self.resources})"

# Define sensor location data structure
class SensorLocation:
    """Represents a potential sensor location for monitoring components"""
    def __init__(self, id, name, cost):
        self.id = id
        self.name = name
        self.cost = cost  # Installation cost in thousands
    
    def __repr__(self):
        return f"Sensor({self.id}: {self.name}, Cost=${self.cost}k)"

print("Data structures defined for components and sensors")

Data structures defined for components and sensors


Data structures defined for components and sensors


Data structures defined for components and sensors


Data structures defined for components and sensors


Data structures defined for components and sensors


In [ ]:
# Initialize the concrete example from the source material
# Define the 3 terminal components
components = [
    Component(1, "Quay Crane", value=85, cost=120, complexity=3, resources=40),
    Component(2, "Yard Block", value=65, cost=80, complexity=2, resources=25),
    Component(3, "Gate System", value=75, cost=95, complexity=4, resources=35)
]

# Define the 4 potential sensor locations
sensor_locations = [
    SensorLocation(1, "Quay Area Sensor", cost=30),
    SensorLocation(2, "Yard Block Sensor", cost=25),
    SensorLocation(3, "Gate System Sensor", cost=35),
    SensorLocation(4, "Perimeter Sensor", cost=40)
]

# Define coverage matrix A[i][l] = 1 if sensor l can monitor component i
coverage_matrix = np.array([
    [1, 1, 0, 0],  # Quay Crane coverage by sensors 1,2
    [0, 1, 1, 0],  # Yard Block coverage by sensors 2,3
    [0, 0, 1, 1]   # Gate System coverage by sensors 3,4
])

# Problem constraints
budget_limit = 250  # Budget in thousands
resource_limit = 80  # Resource limit in person-hours
complexity_limit = 10  # Maximum system complexity
min_value_required = 100  # Minimum operational value required
lambda_penalty = 0.01  # Complexity penalty parameter

print("Problem instance initialized:")
print(f"Components: {len(components)}")
print(f"Sensor locations: {len(sensor_locations)}")
print(f"Budget limit: ${budget_limit}k")
print(f"Resource limit: {resource_limit} person-hours")
print(f"Complexity limit: {complexity_limit}")
print("\nComponents:")
for comp in components:
    print(f"  {comp}")
print("\nSensors:")
for sensor in sensor_locations:
    print(f"  {sensor}")

In [3]:
# Mathematical formulation solver using exhaustive search for small instances
class DigitalTwinScopingSolver:
    """Solves the Terminal Digital Twin Scoping Problem using exact methods"""
    
    def __init__(self, components, sensors, coverage_matrix, 
                 budget_limit, resource_limit, complexity_limit, 
                 min_value_required, lambda_penalty=0.01):
        self.components = components
        self.sensors = sensors
        self.coverage_matrix = coverage_matrix
        self.budget_limit = budget_limit
        self.resource_limit = resource_limit
        self.complexity_limit = complexity_limit
        self.min_value_required = min_value_required
        self.lambda_penalty = lambda_penalty
        self.n_components = len(components)
        self.n_sensors = len(sensors)
    
    def evaluate_solution(self, selected_components, selected_sensors):
        """Evaluate a solution and return objective value and feasibility"""
        # Calculate total values and costs
        total_value = sum(self.components[i].value for i in selected_components)
        total_cost = sum(self.components[i].cost for i in selected_components)
        total_cost += sum(self.sensors[j].cost for j in selected_sensors)
        total_resources = sum(self.components[i].resources for i in selected_components)
        total_complexity = sum(self.components[i].complexity for i in selected_components)
        
        # Check constraints
        budget_feasible = total_cost <= self.budget_limit
        resource_feasible = total_resources <= self.resource_limit
        complexity_feasible = total_complexity <= self.complexity_limit
        value_feasible = total_value >= self.min_value_required
        
        # Check coverage constraints
        coverage_feasible = True
        for i in selected_components:
            component_covered = any(self.coverage_matrix[i][j] for j in selected_sensors)
            if not component_covered:
                coverage_feasible = False
                break
        
        feasible = all([budget_feasible, resource_feasible, complexity_feasible, 
                       value_feasible, coverage_feasible])
        
        # Calculate objective function: maximize value-to-cost ratio minus complexity penalty
        if total_cost > 0:
            objective = (total_value / total_cost) - self.lambda_penalty * total_complexity
        else:
            objective = -float('inf')
        
        return {
            'objective': objective,
            'total_value': total_value,
            'total_cost': total_cost,
            'total_resources': total_resources,
            'total_complexity': total_complexity,
            'feasible': feasible,
            'budget_feasible': budget_feasible,
            'resource_feasible': resource_feasible,
            'complexity_feasible': complexity_feasible,
            'value_feasible': value_feasible,
            'coverage_feasible': coverage_feasible
        }
    
    def find_minimal_sensors(self, selected_components):
        """Find minimal cost sensor set to cover selected components"""
        minimal_sensors = None
        minimal_cost = float('inf')
        
        # Try all sensor combinations
        for r in range(1, self.n_sensors + 1):
            for sensor_combo in combinations(range(self.n_sensors), r):
                # Check if this sensor set covers all selected components
                covers_all = True
                for i in selected_components:
                    covered = any(self.coverage_matrix[i][j] for j in sensor_combo)
                    if not covered:
                        covers_all = False
                        break
                
                if covers_all:
                    sensor_cost = sum(self.sensors[j].cost for j in sensor_combo)
                    if sensor_cost < minimal_cost:
                        minimal_cost = sensor_cost
                        minimal_sensors = list(sensor_combo)
        
        return minimal_sensors if minimal_sensors is not None else []
    
    def solve_exhaustive(self):
        """Solve using exhaustive search for small instances"""
        best_solution = None
        best_objective = -float('inf')
        all_solutions = []
        
        # Try all possible component selections
        for r in range(1, self.n_components + 1):
            for component_combo in combinations(range(self.n_components), r):
                # Find minimal sensors for this component selection
                selected_sensors = self.find_minimal_sensors(list(component_combo))
                
                # Evaluate this solution
                solution = self.evaluate_solution(list(component_combo), selected_sensors)
                solution['selected_components'] = list(component_combo)
                solution['selected_sensors'] = selected_sensors
                
                all_solutions.append(solution)
                
                # Update best solution
                if solution['feasible'] and solution['objective'] > best_objective:
                    best_objective = solution['objective']
                    best_solution = solution
        
        return best_solution, all_solutions

print("Mathematical solver defined")

Mathematical solver defined


Mathematical solver defined


Mathematical solver defined


Mathematical solver defined


Mathematical solver defined


In [ ]:
# Solve the concrete example using the mathematical formulation
solver = DigitalTwinScopingSolver(
    components=components,
    sensors=sensor_locations,
    coverage_matrix=coverage_matrix,
    budget_limit=budget_limit,
    resource_limit=resource_limit,
    complexity_limit=complexity_limit,
    min_value_required=min_value_required,
    lambda_penalty=lambda_penalty
)

# Find optimal solution
best_solution, all_solutions = solver.solve_exhaustive()

print("=== MATHEMATICAL FORMULATION RESULTS ===")
print(f"\nOptimal Solution Found: {best_solution is not None}")

if best_solution:
    print("\n--- OPTIMAL SOLUTION ---")
    print("Selected Components:")
    for i in best_solution['selected_components']:
        comp = components[i]
        print(f"  - {comp.name}: Value={comp.value}, Cost=${comp.cost}k, Complexity={comp.complexity}, Resources={comp.resources}")
    
    print("\nSelected Sensors:")
    for j in best_solution['selected_sensors']:
        sensor = sensor_locations[j]
        print(f"  - {sensor.name}: Cost=${sensor.cost}k")
    
    print(f"\nSolution Summary:")
    print(f"  Total Operational Value: {best_solution['total_value']}")
    print(f"  Total Implementation Cost: ${best_solution['total_cost']}k")
    print(f"  Total Resources Used: {best_solution['total_resources']} person-hours")
    print(f"  Total System Complexity: {best_solution['total_complexity']}")
    print(f"  Value-to-Cost Ratio: {best_solution['total_value']/best_solution['total_cost']:.3f}")
    print(f"  Objective Function Value: {best_solution['objective']:.4f}")
    
    print(f"\nConstraint Satisfaction:")
    print(f"  Budget Constraint: {best_solution['total_cost']}k <= {budget_limit}k ✓" if best_solution['budget_feasible'] else f"  Budget Constraint: VIOLATED")
    print(f"  Resource Constraint: {best_solution['total_resources']} <= {resource_limit} ✓" if best_solution['resource_feasible'] else f"  Resource Constraint: VIOLATED")
    print(f"  Complexity Constraint: {best_solution['total_complexity']} <= {complexity_limit} ✓" if best_solution['complexity_feasible'] else f"  Complexity Constraint: VIOLATED")
    print(f"  Value Requirement: {best_solution['total_value']} >= {min_value_required} ✓" if best_solution['value_feasible'] else f"  Value Requirement: VIOLATED")
    print(f"  Coverage Constraint: SATISFIED ✓" if best_solution['coverage_feasible'] else f"  Coverage Constraint: VIOLATED")
else:
    print("No feasible solution found within the given constraints!")

print(f"\nTotal solutions evaluated: {len(all_solutions)}")
feasible_solutions = [s for s in all_solutions if s['feasible']]
print(f"Feasible solutions: {len(feasible_solutions)}")

In [ ]:
# Visualize the solution space and optimal selection
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle('Terminal Digital Twin Scoping - Mathematical Formulation Results', fontsize=16, fontweight='bold')

# Plot 1: Solution Space - Value vs Cost
ax1 = axes[0, 0]
feasible_costs = [s['total_cost'] for s in feasible_solutions]
feasible_values = [s['total_value'] for s in feasible_solutions]
infeasible_costs = [s['total_cost'] for s in all_solutions if not s['feasible']]
infeasible_values = [s['total_value'] for s in all_solutions if not s['feasible']]

ax1.scatter(infeasible_costs, infeasible_values, c='red', alpha=0.5, s=30, label='Infeasible')
ax1.scatter(feasible_costs, feasible_values, c='green', alpha=0.7, s=50, label='Feasible')
if best_solution:
    ax1.scatter(best_solution['total_cost'], best_solution['total_value'], 
               c='gold', s=200, marker='*', edgecolors='black', linewidth=2, label='Optimal')
ax1.set_xlabel('Total Implementation Cost ($k)')
ax1.set_ylabel('Total Operational Value')
ax1.set_title('Solution Space: Value vs Cost')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Component Selection Visualization
ax2 = axes[0, 1]
if best_solution:
    selected_comp_indices = best_solution['selected_components']
    comp_names = [comp.name for comp in components]
    comp_values = [comp.value for comp in components]
    colors = ['green' if i in selected_comp_indices else 'lightgray' for i in range(len(components))]
    
    bars = ax2.bar(comp_names, comp_values, color=colors, alpha=0.7, edgecolor='black')
    ax2.set_ylabel('Operational Value')
    ax2.set_title('Component Selection (Green = Selected)')
    ax2.tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for bar, value in zip(bars, comp_values):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 1,
                f'{value}', ha='center', va='bottom', fontweight='bold')
else:
    ax2.text(0.5, 0.5, 'No Feasible Solution', ha='center', va='center', transform=ax2.transAxes)
    ax2.set_title('Component Selection')

# Plot 3: Constraint Utilization
ax3 = axes[1, 0]
if best_solution:
    constraints = ['Budget', 'Resources', 'Complexity', 'Value']
    used_values = [
        best_solution['total_cost'],
        best_solution['total_resources'],
        best_solution['total_complexity'],
        best_solution['total_value']
    ]
    limit_values = [budget_limit, resource_limit, complexity_limit, float('inf')]
    
    # Normalize for visualization (except value which is a minimum requirement)
    utilization = []
    for used, limit in zip(used_values, limit_values):
        if limit == float('inf'):  # Value constraint
            utilization.append(min_value_required / used if used > 0 else 0)
        else:
            utilization.append(used / limit if limit > 0 else 0)
    
    colors = ['green' if u <= 1.0 else 'red' for u in utilization]
    bars = ax3.bar(constraints, utilization, color=colors, alpha=0.7, edgecolor='black')
    ax3.axhline(y=1.0, color='red', linestyle='--', alpha=0.7, label='Limit')
    ax3.set_ylabel('Utilization Ratio')
    ax3.set_title('Constraint Utilization (Green = Satisfied)')
    ax3.legend()
    
    # Add percentage labels
    for bar, util in zip(bars, utilization):
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{util:.1%}', ha='center', va='bottom', fontweight='bold')
else:
    ax3.text(0.5, 0.5, 'No Feasible Solution', ha='center', va='center', transform=ax3.transAxes)
    ax3.set_title('Constraint Utilization')

# Plot 4: Coverage Matrix Visualization
ax4 = axes[1, 1]
if best_solution:
    # Create annotated heatmap
    im = ax4.imshow(coverage_matrix, cmap='Blues', alpha=0.7)
    
    # Set ticks and labels
    ax4.set_xticks(range(len(sensor_locations)))
    ax4.set_yticks(range(len(components)))
    ax4.set_xticklabels([f"S{i+1}" for i in range(len(sensor_locations))])
    ax4.set_yticklabels([comp.name[:8] for comp in components])
    
    # Add coverage values
    for i in range(len(components)):
        for j in range(len(sensor_locations)):
            text = ax4.text(j, i, coverage_matrix[i][j],
                           ha="center", va="center", color="black", fontweight="bold")
    
    # Highlight selected sensors and components
    selected_comps = best_solution['selected_components']
    selected_sensors = best_solution['selected_sensors']
    
    # Add borders around selected items
    for i in selected_comps:
        ax4.add_patch(plt.Rectangle((-0.5, i-0.5), len(sensor_locations), 1, 
                                   fill=False, edgecolor='green', linewidth=2))
    for j in selected_sensors:
        ax4.add_patch(plt.Rectangle((j-0.5, -0.5), 1, len(components), 
                                   fill=False, edgecolor='blue', linewidth=2))
    
    ax4.set_title('Coverage Matrix (Green=Selected Comp, Blue=Selected Sensor)')
    ax4.set_xlabel('Sensor Locations')
    ax4.set_ylabel('Components')
else:
    ax4.text(0.5, 0.5, 'No Feasible Solution', ha='center', va='center', transform=ax4.transAxes)
    ax4.set_title('Coverage Matrix')

plt.tight_layout()
plt.show()

In [ ]:
# Sensitivity analysis: How do changes in parameters affect the optimal solution?
def sensitivity_analysis():
    """Perform sensitivity analysis on key parameters"""
    
    # Parameter ranges for sensitivity analysis
    budget_range = [200, 250, 300, 350, 400]
    resource_range = [60, 80, 100, 120, 140]
    lambda_range = [0.005, 0.01, 0.02, 0.05, 0.1]
    
    results = {
        'budget_sensitivity': [],
        'resource_sensitivity': [],
        'lambda_sensitivity': []
    }
    
    # Budget sensitivity
    print("=== BUDGET SENSITIVITY ANALYSIS ===")
    for budget in budget_range:
        solver_budget = DigitalTwinScopingSolver(
            components=components,
            sensors=sensor_locations,
            coverage_matrix=coverage_matrix,
            budget_limit=budget,
            resource_limit=resource_limit,
            complexity_limit=complexity_limit,
            min_value_required=min_value_required,
            lambda_penalty=lambda_penalty
        )
        solution, _ = solver_budget.solve_exhaustive()
        
        if solution:
            results['budget_sensitivity'].append({
                'budget': budget,
                'objective': solution['objective'],
                'value': solution['total_value'],
                'cost': solution['total_cost'],
                'n_components': len(solution['selected_components']),
                'n_sensors': len(solution['selected_sensors'])
            })
            print(f"Budget ${budget}k: Value={solution['total_value']}, Cost=${solution['total_cost']}k, Components={len(solution['selected_components'])}")
        else:
            results['budget_sensitivity'].append({
                'budget': budget, 'objective': 0, 'value': 0, 'cost': 0, 'n_components': 0, 'n_sensors': 0
            })
            print(f"Budget ${budget}k: No feasible solution")
    
    # Resource sensitivity
    print("\n=== RESOURCE SENSITIVITY ANALYSIS ===")
    for resources in resource_range:
        solver_resource = DigitalTwinScopingSolver(
            components=components,
            sensors=sensor_locations,
            coverage_matrix=coverage_matrix,
            budget_limit=budget_limit,
            resource_limit=resources,
            complexity_limit=complexity_limit,
            min_value_required=min_value_required,
            lambda_penalty=lambda_penalty
        )
        solution, _ = solver_resource.solve_exhaustive()
        
        if solution:
            results['resource_sensitivity'].append({
                'resources': resources,
                'objective': solution['objective'],
                'value': solution['total_value'],
                'cost': solution['total_cost'],
                'n_components': len(solution['selected_components'])
            })
            print(f"Resources {resources}: Value={solution['total_value']}, Cost=${solution['total_cost']}k, Components={len(solution['selected_components'])}")
        else:
            results['resource_sensitivity'].append({
                'resources': resources, 'objective': 0, 'value': 0, 'cost': 0, 'n_components': 0
            })
            print(f"Resources {resources}: No feasible solution")
    
    # Lambda (complexity penalty) sensitivity
    print("\n=== COMPLEXITY PENALTY SENSITIVITY ANALYSIS ===")
    for lambda_val in lambda_range:
        solver_lambda = DigitalTwinScopingSolver(
            components=components,
            sensors=sensor_locations,
            coverage_matrix=coverage_matrix,
            budget_limit=budget_limit,
            resource_limit=resource_limit,
            complexity_limit=complexity_limit,
            min_value_required=min_value_required,
            lambda_penalty=lambda_val
        )
        solution, _ = solver_lambda.solve_exhaustive()
        
        if solution:
            results['lambda_sensitivity'].append({
                'lambda': lambda_val,
                'objective': solution['objective'],
                'value': solution['total_value'],
                'cost': solution['total_cost'],
                'complexity': solution['total_complexity'],
                'n_components': len(solution['selected_components'])
            })
            print(f"Lambda {lambda_val}: Value={solution['total_value']}, Complexity={solution['total_complexity']}, Components={len(solution['selected_components'])}")
        else:
            results['lambda_sensitivity'].append({
                'lambda': lambda_val, 'objective': 0, 'value': 0, 'cost': 0, 'complexity': 0, 'n_components': 0
            })
            print(f"Lambda {lambda_val}: No feasible solution")
    
    return results

# Run sensitivity analysis
sensitivity_results = sensitivity_analysis()

In [ ]:
# Visualize sensitivity analysis results
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Sensitivity Analysis - Terminal Digital Twin Scoping', fontsize=16, fontweight='bold')

# Budget sensitivity plot
ax1 = axes[0, 0]
budget_data = sensitivity_results['budget_sensitivity']
budgets = [d['budget'] for d in budget_data]
values = [d['value'] for d in budget_data]
components = [d['n_components'] for d in budget_data]

ax1_twin = ax1.twinx()
line1 = ax1.plot(budgets, values, 'b-o', linewidth=2, markersize=8, label='Total Value')
line2 = ax1_twin.plot(budgets, components, 'r-s', linewidth=2, markersize=8, label='Components Selected')

ax1.set_xlabel('Budget Limit ($k)')
ax1.set_ylabel('Total Operational Value', color='b')
ax1_twin.set_ylabel('Number of Components', color='r')
ax1.set_title('Budget Sensitivity Analysis')
ax1.grid(True, alpha=0.3)

# Combine legends
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left')

# Resource sensitivity plot
ax2 = axes[0, 1]
resource_data = sensitivity_results['resource_sensitivity']
resources = [d['resources'] for d in resource_data]
values = [d['value'] for d in resource_data]
components = [d['n_components'] for d in resource_data]

ax2_twin = ax2.twinx()
line1 = ax2.plot(resources, values, 'b-o', linewidth=2, markersize=8, label='Total Value')
line2 = ax2_twin.plot(resources, components, 'r-s', linewidth=2, markersize=8, label='Components Selected')

ax2.set_xlabel('Resource Limit (person-hours)')
ax2.set_ylabel('Total Operational Value', color='b')
ax2_twin.set_ylabel('Number of Components', color='r')
ax2.set_title('Resource Sensitivity Analysis')
ax2.grid(True, alpha=0.3)

# Lambda (complexity penalty) sensitivity plot
ax3 = axes[1, 0]
lambda_data = sensitivity_results['lambda_sensitivity']
lambdas = [d['lambda'] for d in lambda_data]
values = [d['value'] for d in lambda_data]
complexities = [d['complexity'] for d in lambda_data]

ax3_twin = ax3.twinx()
line1 = ax3.plot(lambdas, values, 'b-o', linewidth=2, markersize=8, label='Total Value')
line2 = ax3_twin.plot(lambdas, complexities, 'g-^', linewidth=2, markersize=8, label='System Complexity')

ax3.set_xlabel('Complexity Penalty (λ)')
ax3.set_ylabel('Total Operational Value', color='b')
ax3_twin.set_ylabel('System Complexity', color='g')
ax3.set_title('Complexity Penalty Sensitivity Analysis')
ax3.grid(True, alpha=0.3)
ax3.set_xscale('log')

# Combine legends
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax3.legend(lines, labels, loc='upper right')

# Solution quality comparison across different scenarios
ax4 = axes[1, 1]
scenarios = ['Base Case', 'High Budget', 'High Resources', 'Low Complexity Penalty']
objectives = [
    best_solution['objective'] if best_solution else 0,
    budget_data[-1]['objective'] if budget_data[-1]['objective'] > 0 else 0,
    resource_data[-1]['objective'] if resource_data[-1]['objective'] > 0 else 0,
    lambda_data[0]['objective'] if lambda_data[0]['objective'] > 0 else 0
]

colors = ['green', 'lightblue', 'lightgreen', 'gold']
bars = ax4.bar(scenarios, objectives, color=colors, alpha=0.7, edgecolor='black')
ax4.set_ylabel('Objective Function Value')
ax4.set_title('Solution Quality Across Scenarios')
ax4.tick_params(axis='x', rotation=45)
ax4.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, obj in zip(bars, objectives):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height + max(objectives)*0.01,
            f'{obj:.4f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

### Why this Tier exists vs earlier Tiers
This is the foundational Tier that establishes the mathematical framework for the Terminal Digital Twin Scoping Problem. It provides:
- **Exact optimal solutions** through mathematical programming
- **Rigorous constraint handling** ensuring all operational requirements are met
- **Theoretical foundation** for understanding the problem structure and trade-offs
- **Benchmark solutions** against which heuristic methods can be compared

### Pros / Cons vs other approaches
**Pros:**
- Guarantees optimal solution for small instances
- Provides provable bounds on solution quality
- Handles complex constraints systematically
- Enables sensitivity analysis for parameter tuning

**Cons:**
- Computationally expensive for large instances (exponential complexity)
- Requires complete problem specification
- Less flexible for dynamic or uncertain environments
- May be overkill for quick decision-making scenarios

### When to use this Tier
- **Strategic planning** where optimality is critical
- **Small to medium instances** where exhaustive search is feasible
- **Benchmark development** to evaluate heuristic methods
- **Parameter sensitivity analysis** for understanding system behavior
- **Academic research** requiring theoretical guarantees

### Key Insights from Mathematical Formulation
The mathematical formulation reveals several critical insights about digital twin scoping:
1. **Interdependency constraints** significantly impact feasible solutions
2. **Coverage requirements** create complex combinatorial challenges
3. **Multi-objective nature** requires careful balance between value, cost, and complexity
4. **Parameter sensitivity** shows budget constraints are often the most binding limitation
5. **Solution structure** typically favors high-value, low-complexity components with minimal sensor requirements